In [ ]:
import tensorflow as tf
import random
import numpy as np
import sys
sys.path.insert(0, "/home/dajiang/smart-pixels-ml/two_bit_optimization_helpers") # Change this line to wherever the two_bit_optimization_helpers is located 
from prepare_tfrecords import generate_tfrecords, load_tfrecords
from train import create_model, train, get_best_thresholds, cleanup_models_and_generators, save_performance_parquet

##### Required arguments

In [ ]:
### REQUIRED/MIGHT NEED TO CHANGE ###
dataset_dir='/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets'
weights_directory='/data/dajiang/smart-pixels/weights/dataset_3src_16x16_50x12P5_centeredIncidence_weights/'
performance_directory='/home/dajiang/smart-pixels-ml/processed_parquets/dataset_3src_16x16_50x12P5_centeredIncidence/test_dataset_3src_16x16_50x12P5_centeredIncidence/2bit_optimized/'

# For non-quantized models: Conv2D_Max, Conv2D_Full, Conv2D_Slim, Conv1D_Full, Conv1D_Slim, Mlp_Full, Mlp_Slim
# For quantized models: QConv2D_Max, QConv2D_Full, QConv2D_Slim, QConv1D_Full, QConv1D_Slim, QMlp_Full, QMlp_Slim
model_type='Conv2D_Max' 

# if you already generated the TFRecords previously, set to True or you will redo it and overwrite the existing ones (saves time)
tfrecords_exist=False

select_contained=True
initial_thresholds=[247.8, 668.4, 1662.9]
threshold_offset=80.0
seed=10

### DON'T CHANGE UNLESS YOU KNOW WHAT YOU ARE DOING ###
train_batch_size=5000
val_batch_size=5000
noise1=[0,80]
noise2=-1
epochs1=2000
epochs2=1000
timeslices=2
train_type1='soft_quantize_layer' # soft_quantize_layer or 2bit_optimized
train_type2='2bit_optimized' # soft_quantize_layer or 2bit_optimized
soft_quantize_layer1=True # soft_quantize_layer
soft_quantize_layer2=False # 2bit_optimized
initial_levels=np.array([0.0, 1.0, 2.0, 3.0], dtype=np.float32)

##### Set random seeds for tensorflow and random

In [ ]:
tf.random.set_seed(seed)
random.seed(seed)

##### Generating and saving TFRecords and then loading the TFRecords with noise added

In [ ]:
dataset_train_dir, dataset_validation_dir, tfrecords_dir_train, tfrecords_dir_val = generate_tfrecords(
    dataset_dir=dataset_dir,
    model_type=model_type,
    train_batch_size=train_batch_size,
    val_batch_size=val_batch_size,
    select_contained=select_contained,
    timeslices=timeslices,
    tfrecords_exist=tfrecords_exist,
    seed=seed,
)

training_generator1, validation_generator1 = load_tfrecords(
    tfrecords_dir_train, 
    tfrecords_dir_val,
    noise=noise1,
    seed=seed,
)

##### Training part 1

In [ ]:
model1 = create_model(
    model_type=model_type,
    timeslices=timeslices,
    soft_quantize_layer=soft_quantize_layer1,
    initial_thresholds=initial_thresholds,
    threshold_offset=threshold_offset,
    initial_levels=initial_levels,
)

checkpoints_directory1, fingerprint1 = train(
    model=model1,
    model_type=model_type, 
    weights_directory=weights_directory,
    training_generator=training_generator1,
    validation_generator=validation_generator1, 
    timeslices=timeslices,
    train_type=train_type1,
    epochs=epochs1,
    seed=seed, 
)

##### Input Digitization

In [ ]:
thresholds, levels = get_best_thresholds(
    checkpoints=checkpoints_directory1,
    dataset_train_dir=dataset_train_dir,
    dataset_validation_dir=dataset_validation_dir,
    model_type=model_type,
    timeslices=timeslices,
    initial_thresholds=initial_thresholds,
    threshold_offset=threshold_offset,
    initial_levels=initial_levels,
)

##### Cleanup of previous models and generators

In [ ]:
cleanup_models_and_generators([model1, training_generator1, validation_generator1])

##### Load TFRecords without noise added, then digitized to 2bits according to thresholds and levels from part 1

In [ ]:
training_generator2, validation_generator2 = load_tfrecords(
    tfrecords_dir_train, 
    tfrecords_dir_val,
    noise=noise2,
    digitize=True,
    digitize_levels=levels,
    digitize_thresholds=thresholds,
    seed=seed,
)

##### Reset random seeds for tensorflow and random

In [ ]:
tf.random.set_seed(seed)
random.seed(seed)

##### Training part 2

In [ ]:
model2 = create_model(
    model_type=model_type,
    timeslices=timeslices,
    soft_quantize_layer=soft_quantize_layer2,
)

checkpoints_directory2, fingerprint2 = train(
    model=model2,
    model_type=model_type, 
    weights_directory=weights_directory,
    training_generator=training_generator2,
    validation_generator=validation_generator2, 
    timeslices=timeslices,
    train_type=train_type2,
    epochs=epochs2,
    seed=seed, 
)

##### Save best weights from part 2 and process to parquet files with performance variables

In [ ]:
save_performance_parquet(
    checkpoints=checkpoints_directory2,
    output_directory=performance_directory,
    test_generator=validation_generator2, 
    model_type=model_type,
    train_type=train_type2,
    fingerprint=fingerprint2,
    timeslices=2,
    soft_quantize_layer=False,
)